# Chapitre 8 - Cours 4 : pauses de processus, majorité, baux et jetons de clôture

Un nœud qui détient un **bail** (un verrou à durée limitée) peut s'endormir au pire moment
et se réveiller sans savoir qu'il a dormi, alors que les autres l'ont déclaré mort et
ont donné son bail à un autre. Ce notebook montre pourquoi cela arrive, pourquoi un nœud
ne peut pas se fier à son propre jugement, et comment le **jeton de clôture** empêche un
tel « zombie » de corrompre les données. Il couvre la section *Process Pauses* (*Response time guarantees*,
*Limiting the impact of garbage collection*) et le début de *Knowledge, Truth, and Lies* :
*The Truth Is Defined by the Majority* (*The leader and the lock*, *Fencing tokens*).

**Plan.**

1. Le bail et la boucle naïve : un processus peut s'arrêter n'importe où.
2. Garanties de temps de réponse et ramasse-miettes.
3. La majorité fait loi.
4. Verrous et baux distribués : la corruption par un zombie.
5. Les jetons de clôture.

In [1]:
import os
import signal
import subprocess
import sys
import textwrap
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import distsys as ds
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, set_style, subtitle)
from ddia.requirements import plain_log_axis
from ddia.sim import Network
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 4)
rng = np.random.default_rng(0)